# LeetCode #155 — Min Stack: ngăn xếp lấy min trong `O(1)`

**Mức độ:** Medium · **Chủ đề:** Stack, Design

Skeleton trên LeetCode:

```python
class MinStack:
    def __init__(self):
    def push(self, val: int) -> None:
    def pop(self) -> None:
    def top(self) -> int:
    def getMin(self) -> int:
```

Mục tiêu của notebook:

1. Thiết kế cấu trúc dữ liệu: mọi thao tác `O(1)`.
2. Ý tưởng then chốt: lưu **min tại thời điểm push** cùng với mỗi phần tử — vì ngăn xếp chỉ bỏ phần tử **trên cùng**, min "của quá khứ" luôn còn hợp lệ.
3. Hai cách cài: ngăn xếp cặp `(val, min)` và hai ngăn xếp (ngăn min chỉ push khi `val <= min`).
4. Bonus: cách lưu **hiệu** `val − min` dùng `O(1)` bộ nhớ phụ.

# 1. Bài toán là gì?

Thiết kế ngăn xếp hỗ trợ:

- `push(val)`: đẩy `val` vào.
- `pop()`: bỏ phần tử trên cùng.
- `top()`: trả phần tử trên cùng.
- `getMin()`: trả phần tử **nhỏ nhất** hiện có.

Mọi thao tác phải `O(1)` thời gian.

**Constraints (theo đề):**

- `-2^31 <= val <= 2^31 - 1`
- `pop`, `top`, `getMin` chỉ được gọi khi ngăn xếp **không rỗng**.
- Tối đa `3 * 10^4` lời gọi.

---

## Ví dụ

```text
["MinStack","push","push","push","getMin","pop","top","getMin"]
[[],[-2],[0],[-3],[],[],[],[]]
->
[null,null,null,null,-3,null,0,-2]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Ngăn xếp (stack)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Ngăn xếp (stack)

### Ngăn xếp là gì?

Ngăn xếp là cấu trúc **vào sau, ra trước** (LIFO — Last In, First Out), giống chồng đĩa: chỉ đặt thêm hoặc lấy ra ở **đỉnh**.

```text
push 1   push 2   push 3   pop → 3   top = 2
                   │ 3 │
          │ 2 │    │ 2 │    │ 2 │
 │ 1 │    │ 1 │    │ 1 │    │ 1 │
 └───┘    └───┘    └───┘    └───┘
```

| Thao tác | Ý nghĩa | Python (`list`) | Độ phức tạp |
|---|---|---|---|
| `push(x)` | đặt `x` lên đỉnh | `st.append(x)` | `O(1)` |
| `pop()` | lấy và bỏ đỉnh | `st.pop()` | `O(1)` |
| `top()` | xem đỉnh | `st[-1]` | `O(1)` |
| rỗng? | | `not st` | `O(1)` |

In [1]:
st = []
for x in [1, 2, 3]:
    st.append(x)
    print("push", x, "->", st)
print("top:", st[-1])
while st:
    print("pop", st.pop(), "->", st)
try:
    st.pop()
except IndexError as e:
    print("pop khi rỗng:", e)

push 1 -> [1]
push 2 -> [1, 2]
push 3 -> [1, 2, 3]
top: 3
pop 3 -> [1, 2]
pop 2 -> [1]
pop 1 -> []
pop khi rỗng: pop from empty list


### Khi nào nghĩ tới ngăn xếp?

Khi thứ được mở **sau cùng** phải được đóng/xử lý **trước tiên**:

- Dấu ngoặc lồng nhau: `( [ { } ] )`.
- Đường dẫn thư mục: `..` hủy thư mục vừa vào.
- Tính biểu thức hậu tố, gọi hàm đệ quy (máy tính dùng *call stack*).
- "Hoàn tác" (undo).

Ví dụ kinh điển — kiểm tra ngoặc hợp lệ: gặp ngoặc mở thì push; gặp ngoặc đóng thì đỉnh phải là ngoặc mở tương ứng.

In [2]:
def ngoac_hop_le(s):
    cap = {")": "(", "]": "[", "}": "{"}
    st = []
    for ch in s:
        if ch in "([{":
            st.append(ch)
        else:
            if not st or st[-1] != cap[ch]:
                return False
            st.pop()
    return not st                        # còn ngoặc mở chưa đóng → sai

for s in ["([]{})", "([)]", "((", "}"]:
    print(f"{s!r:>9}: {ngoac_hop_le(s)}")

 '([]{})': True
   '([)]': False
     '((': False
      '}': False


### Lỗi hay gặp

- `pop()` / `st[-1]` khi ngăn xếp rỗng → `IndexError`. Luôn kiểm tra `if st`.
- Quên kiểm tra ngăn xếp **còn sót** phần tử ở cuối.
- Dùng `list.pop(0)` / `insert(0, x)` (đầu danh sách) — đó là hàng đợi và tốn `O(n)`; ngăn xếp luôn thao tác ở **cuối** list.

## 2.2 Áp dụng vào bài này

Thiết kế **ngăn xếp** có thêm `getMin` `O(1)`. Vì ngăn xếp chỉ bỏ phần tử ở **đỉnh**, ta lưu kèm mỗi phần tử **giá trị min tại thời điểm push** — khi phần tử đó ở đỉnh, min ấy vẫn đúng.

# 3. Nền tảng

In [3]:
from typing import List, Tuple
import random, time

st = []
st.append(5); st.append(2)
print("top:", st[-1], "| pop:", st.pop(), "| còn:", st)

top: 2 | pop: 2 | còn: [5]


Baseline ngây thơ: `getMin` gọi `min(self.stack)` — `O(n)`. Vấn đề: sau `pop` phần tử nhỏ nhất, phải tìm min **mới** — làm sao biết nhanh?

In [4]:
class MinStackNaive:
    def __init__(self):
        self.st = []
    def push(self, val):
        self.st.append(val)
    def pop(self):
        self.st.pop()
    def top(self):
        return self.st[-1]
    def getMin(self):
        return min(self.st)                     # O(n)

# 4. Ý tưởng chính

Ngăn xếp chỉ bỏ phần tử **trên cùng**. Khi phần tử thứ `i` (tính từ đáy) đang ở đỉnh, ngăn xếp chứa **đúng** các phần tử `0..i` — giống hệt lúc vừa push phần tử `i`.

→ Nếu lúc push phần tử `i` ta lưu `min(phần tử 0..i)`, thì bất cứ khi nào nó ở đỉnh, giá trị đó **chính là** min hiện tại.

```text
push(val):  cur_min = val nếu rỗng, ngược lại min(val, min ở đỉnh)
            push (val, cur_min)
getMin():   min ở đỉnh
```

In [5]:
class MinStackPairs:
    def __init__(self):
        self.st: List[Tuple[int, int]] = []     # (val, min tới đây)

    def push(self, val: int) -> None:
        cur = val if not self.st else min(val, self.st[-1][1])
        self.st.append((val, cur))

    def pop(self) -> None:
        self.st.pop()

    def top(self) -> int:
        return self.st[-1][0]

    def getMin(self) -> int:
        return self.st[-1][1]

# 5. Mô phỏng

In [6]:
def run_ops(cls, ops, args):
    obj, out = None, []
    for op, a in zip(ops, args):
        if op in ("MinStack", cls.__name__):
            obj = cls(); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


def trace_pairs(ops, args):
    obj = MinStackPairs()
    for op, a in zip(ops[1:], args[1:]):
        r = getattr(obj, op)(*a)
        shown = f"{op}({', '.join(map(str, a))})"
        res = "" if r is None else f" -> {r}"
        print(f"{shown:<10}{res:<6} | ngăn xếp (val, min): {obj.st}")

ops = ["MinStack", "push", "push", "push", "getMin", "pop", "top", "getMin"]
args = [[], [-2], [0], [-3], [], [], [], []]
trace_pairs(ops, args)

push(-2)         | ngăn xếp (val, min): [(-2, -2)]
push(0)          | ngăn xếp (val, min): [(-2, -2), (0, -2)]
push(-3)         | ngăn xếp (val, min): [(-2, -2), (0, -2), (-3, -3)]
getMin()   -> -3 | ngăn xếp (val, min): [(-2, -2), (0, -2), (-3, -3)]
pop()            | ngăn xếp (val, min): [(-2, -2), (0, -2)]
top()      -> 0  | ngăn xếp (val, min): [(-2, -2), (0, -2)]
getMin()   -> -2 | ngăn xếp (val, min): [(-2, -2), (0, -2)]


# 6. Cách 2: hai ngăn xếp, ngăn min "nén"

Ngăn `mins` chỉ push khi `val <= mins[-1]` (dùng `<=` để xử lý **trùng**); khi pop, nếu giá trị bị pop bằng `mins[-1]` thì pop `mins` theo. Tiết kiệm bộ nhớ khi min ít thay đổi.

In [7]:
class MinStackTwo:
    def __init__(self):
        self.st: List[int] = []
        self.mins: List[int] = []

    def push(self, val: int) -> None:
        self.st.append(val)
        if not self.mins or val <= self.mins[-1]:
            self.mins.append(val)

    def pop(self) -> None:
        if self.st.pop() == self.mins[-1]:
            self.mins.pop()

    def top(self) -> int:
        return self.st[-1]

    def getMin(self) -> int:
        return self.mins[-1]


s2 = MinStackTwo()
for v in [5, 3, 7, 3, 8]:
    s2.push(v)
print("st:", s2.st, "| mins:", s2.mins)

st: [5, 3, 7, 3, 8] | mins: [5, 3, 3]


# 7. Phiên bản đúng format LeetCode

In [8]:
class MinStack:
    def __init__(self):
        self.st = []                            # (val, min tới đây)

    def push(self, val: int) -> None:
        cur = val if not self.st else min(val, self.st[-1][1])
        self.st.append((val, cur))

    def pop(self) -> None:
        self.st.pop()

    def top(self) -> int:
        return self.st[-1][0]

    def getMin(self) -> int:
        return self.st[-1][1]

In [9]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


check("Ví dụ", run_ops(MinStack, ops, args), [None, None, None, None, -3, None, 0, -2])
check("Trùng min", run_ops(MinStack, ["MinStack", "push", "push", "push", "pop", "getMin"],
                           [[], [0], [1], [0], [], []]), [None, None, None, None, None, 0])


def random_ops(k, lo=-5, hi=5):
    ops, args, size = ["MinStack"], [[]], 0
    for _ in range(k):
        r = random.random()
        if size == 0 or r < 0.45:
            ops.append("push"); args.append([random.randint(lo, hi)]); size += 1
        elif r < 0.65:
            ops.append("pop"); args.append([]); size -= 1
        elif r < 0.8:
            ops.append("top"); args.append([])
        else:
            ops.append("getMin"); args.append([])
    return ops, args


random.seed(155)
for _ in range(3000):
    o, a = random_ops(random.randint(1, 40))
    exp = run_ops(MinStackNaive, o, a)
    assert run_ops(MinStack, o, a) == exp == run_ops(MinStackTwo, o, a), (o, a)
print("3000 chuỗi thao tác ngẫu nhiên: khớp bản ngây thơ")

o, a = ["MinStack"], [[]]
for i in range(15000):
    o.append("push"); a.append([15000 - i])
for i in range(15000):
    o.append("getMin"); a.append([])
for name, cls in [("MinStack (cặp)", MinStack), ("Naive min()", MinStackNaive)]:
    t0 = time.time(); run_ops(cls, o, a); t1 = time.time()
    print(f"{name:<15}: 3·10^4 thao tác trong {t1 - t0:.3f}s")

Ví dụ            PASS  [None, None, None, None, -3, None, 0, -2]
Trùng min        PASS  [None, None, None, None, None, 0]
3000 chuỗi thao tác ngẫu nhiên: khớp bản ngây thơ


MinStack (cặp) : 3·10^4 thao tác trong 0.011s

Naive min()    : 3·10^4 thao tác trong 3.287s


# 8. Vì sao đúng? Độ phức tạp?

**Bất biến (cách cặp):** phần tử thứ `i` từ đáy lưu `m_i = min(v_0, …, v_i)`. Ngăn xếp luôn là một **tiền tố** `v_0..v_k` của lịch sử các phần tử còn sống, nên `getMin = m_k` đúng.

**Cách hai ngăn:** `mins` là dãy các "kỷ lục mới hoặc bằng" theo thứ tự push; phần tử đỉnh của `mins` là min của các phần tử đang còn. Dùng `<=` để mỗi bản sao của min đều có mặt trong `mins` — pop một bản thì vẫn còn bản khác.

```text
Mọi thao tác: O(1)
Bộ nhớ: O(n) (cặp: 2n số; hai ngăn: n + số lần lập kỷ lục)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Chỉ giữ **một** biến `min`

In [10]:
class WrongSingleMin:
    def __init__(self):
        self.st, self.m = [], float("inf")
    def push(self, val):
        self.st.append(val); self.m = min(self.m, val)
    def pop(self):
        self.st.pop()                                # SAI: pop mất min thì self.m sai
    def top(self):
        return self.st[-1]
    def getMin(self):
        return self.m

print("Sai :", run_ops(WrongSingleMin, ops, args))
print("Đúng:", run_ops(MinStack, ops, args))

Sai : [None, None, None, None, -3, None, 0, -3]
Đúng: [None, None, None, None, -3, None, 0, -2]


## Lỗi 2: Hai ngăn xếp nhưng chỉ push vào `mins` khi `val < mins[-1]` (thiếu `=`)

In [11]:
class WrongStrict(MinStackTwo):
    def push(self, val):
        self.st.append(val)
        if not self.mins or val < self.mins[-1]:     # SAI: bỏ qua bản sao của min
            self.mins.append(val)

o2 = ["MinStack", "push", "push", "push", "pop", "getMin"]
a2 = [[], [0], [1], [0], [], []]
try:
    print("Sai :", run_ops(WrongStrict, o2, a2))
except IndexError as e:
    print("IndexError:", e)
print("Đúng:", run_ops(MinStack, o2, a2))

IndexError: list index out of range
Đúng: [None, None, None, None, None, 0]


Push `0, 1, 0`: `mins = [0]` (bản `0` thứ hai bị bỏ). Pop `0` trên cùng → bằng `mins[-1]` → pop `mins` → `mins` rỗng dù ngăn xếp vẫn còn `[0, 1]`.

## Lỗi 3: Khởi tạo `min` bằng một số "đủ lớn" như `10**9`

`val` có thể tới `2^31 − 1 > 10^9` → `getMin` trả sai. Dùng phần tử đầu tiên (hoặc `float('inf')`).

## Lỗi 4: Sắp xếp / dùng heap

`heapq` cho min `O(1)` nhưng `pop` của ngăn xếp phải xóa phần tử **bất kỳ** khỏi heap → `O(n)` hoặc phải xóa lười. Không cần phức tạp vậy.

# 10. Bonus: `O(1)` bộ nhớ phụ — lưu hiệu

Chỉ lưu một ngăn `diff` và biến `m` (min hiện tại). Push `val`: lưu `val − m` (với `m` **cũ**), rồi `m = min(m, val)`.

- Nếu hiệu lưu `d < 0` → phần tử này đã lập min mới, và min **trước đó** là `m − d`.
- `top`: nếu `d < 0` thì phần tử là `m`, ngược lại là `m + d`.

Python có số nguyên không giới hạn nên không lo tràn (trong C++/Java cần `long`).

In [12]:
class MinStackDiff:
    def __init__(self):
        self.diff, self.m = [], None

    def push(self, val: int) -> None:
        if not self.diff:
            self.diff.append(0); self.m = val
        else:
            self.diff.append(val - self.m)
            if val < self.m:
                self.m = val

    def pop(self) -> None:
        d = self.diff.pop()
        if d < 0:
            self.m = self.m - d                 # khôi phục min trước đó
        if not self.diff:
            self.m = None

    def top(self) -> int:
        d = self.diff[-1]
        return self.m if d < 0 else self.m + d

    def getMin(self) -> int:
        return self.m


for _ in range(3000):
    o, a = random_ops(random.randint(1, 40), -2**31, 2**31 - 1)
    assert run_ops(MinStackDiff, o, a) == run_ops(MinStackNaive, o, a)
print("MinStackDiff: 3000 chuỗi ngẫu nhiên (giá trị tới ±2^31) khớp")

MinStackDiff: 3000 chuỗi ngẫu nhiên (giá trị tới ±2^31) khớp


# 11. Mẹo nhận diện pattern

- **Truy vấn tổng hợp (min/max/gcd…) trên ngăn xếp** → lưu giá trị tổng hợp **tại thời điểm push**.
- **Hàng đợi có min** → hai ngăn xếp có min (bài tập 4) hoặc deque đơn điệu (#239).

Bài liên quan: **#716 Max Stack** (Hard), **#232 Implement Queue using Stacks**, **#239 Sliding Window Maximum**, **#901 Online Stock Span**, **#1381 Design a Stack With Increment Operation**.

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với cách hai ngăn, ghi `st` và `mins` sau mỗi thao tác: `push 2, push 0, push 3, push 0, getMin, pop, getMin, pop, getMin, pop, getMin`.

## Bài 2 — Điền chỗ trống (cách cặp)

In [13]:
class Exercise2MinStack:
    def __init__(self):
        self.st = []

    def push(self, val: int) -> None:
        # TODO
        pass

    def pop(self) -> None:
        self.st.pop()

    def top(self) -> int:
        return self.st[-1][0]

    def getMin(self) -> int:
        return self.st[-1][1]


# assert run_ops(Exercise2MinStack, ["MinStack", "push", "push", "push", "getMin", "pop", "top", "getMin"], [[], [-2], [0], [-3], [], [], [], []]) == [None, None, None, None, -3, None, 0, -2]

## Bài 3 — Biến thể: Max Stack đơn giản

Ngăn xếp với `push`, `pop` (trả về phần tử), `top`, `peekMax` — tất cả `O(1)`. (Bản LeetCode #716 còn có `popMax`, khó hơn.)

In [14]:
class Exercise3MaxStack:
    def __init__(self):
        # TODO
        pass

    def push(self, x: int) -> None:
        pass

    def pop(self) -> int:
        pass

    def top(self) -> int:
        pass

    def peekMax(self) -> int:
        pass


# m3 = Exercise3MaxStack(); m3.push(5); m3.push(1); m3.push(5); assert m3.peekMax() == 5 and m3.pop() == 5 and m3.peekMax() == 5 and m3.pop() == 1 and m3.top() == 5

## Bài 4 — Biến thể: hàng đợi có `getMin` `O(1)` khấu hao

Dùng **hai** min-stack: `inbox` nhận push, `outbox` để pop; khi `outbox` rỗng thì đổ toàn bộ `inbox` sang. `getMin = min(min của inbox, min của outbox)`.

In [15]:
class Exercise4MinQueue:
    def __init__(self):
        # TODO
        pass

    def push(self, x: int) -> None:
        pass

    def pop(self) -> int:
        pass

    def getMin(self) -> int:
        pass


# q4 = Exercise4MinQueue(); q4.push(3); q4.push(1); q4.push(2); assert q4.getMin() == 1 and q4.pop() == 3 and q4.getMin() == 1 and q4.pop() == 1 and q4.getMin() == 2

## Bài 5 — Mini challenge: min của mọi cửa sổ trượt độ dài `k`

Dùng `Exercise4MinQueue` (hoặc đáp án của nó) để tính min của từng cửa sổ độ dài `k` trong `O(n)` khấu hao.

In [16]:
def exercise_5_sliding_min(nums: List[int], k: int) -> List[int]:
    # TODO
    pass


# assert exercise_5_sliding_min([1, 3, -1, -3, 5, 3, 6, 7], 3) == [-1, -3, -3, -3, 3, 3]
# assert exercise_5_sliding_min([4], 1) == [4]

# 13. Gợi ý

**Bài 1**
- Mức 1: chú ý `0` được push vào `mins` **hai lần** (nhờ `<=`).

**Bài 3**
- Mức 1: giống Min Stack, lưu `(val, max tới đây)`.

**Bài 4**
- Mức 1: mỗi phần tử được chuyển từ `inbox` sang `outbox` đúng một lần → khấu hao `O(1)`.
- Mức 2: khi đổ sang `outbox`, push lại bằng `push` của min-stack để tính min **mới** cho thứ tự mới.

**Bài 5**
- Mức 1: push từng phần tử; khi đủ `k` thì ghi `getMin()` rồi `pop()` phần tử cũ nhất.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [17]:
s1 = MinStackTwo()
for op, a in [("push", 2), ("push", 0), ("push", 3), ("push", 0), ("getMin", None), ("pop", None),
              ("getMin", None), ("pop", None), ("getMin", None), ("pop", None), ("getMin", None)]:
    r = getattr(s1, op)(a) if a is not None else getattr(s1, op)()
    label = f"{op} {a}" if a is not None else op
    print(f"{label:<8} -> {str(r) if r is not None else '':<3} | st = {s1.st}, mins = {s1.mins}")

push 2   ->     | st = [2], mins = [2]
push 0   ->     | st = [2, 0], mins = [2, 0]
push 3   ->     | st = [2, 0, 3], mins = [2, 0]
push 0   ->     | st = [2, 0, 3, 0], mins = [2, 0, 0]
getMin   -> 0   | st = [2, 0, 3, 0], mins = [2, 0, 0]
pop      ->     | st = [2, 0, 3], mins = [2, 0]
getMin   -> 0   | st = [2, 0, 3], mins = [2, 0]
pop      ->     | st = [2, 0], mins = [2, 0]
getMin   -> 0   | st = [2, 0], mins = [2, 0]
pop      ->     | st = [2], mins = [2]
getMin   -> 2   | st = [2], mins = [2]


## Đáp án Bài 2

In [18]:
class Answer2MinStack(MinStack):
    pass

assert run_ops(Answer2MinStack, ["MinStack", "push", "push", "push", "getMin", "pop", "top", "getMin"], [[], [-2], [0], [-3], [], [], [], []]) == [None, None, None, None, -3, None, 0, -2]
print("Bài 2 PASS (giống MinStack ở mục 7)")

Bài 2 PASS (giống MinStack ở mục 7)


## Đáp án Bài 3

In [19]:
class Answer3MaxStack:
    def __init__(self):
        self.st = []

    def push(self, x: int) -> None:
        self.st.append((x, x if not self.st else max(x, self.st[-1][1])))

    def pop(self) -> int:
        return self.st.pop()[0]

    def top(self) -> int:
        return self.st[-1][0]

    def peekMax(self) -> int:
        return self.st[-1][1]

m3 = Answer3MaxStack(); m3.push(5); m3.push(1); m3.push(5); assert m3.peekMax() == 5 and m3.pop() == 5 and m3.peekMax() == 5 and m3.pop() == 1 and m3.top() == 5
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [20]:
class Answer4MinQueue:
    def __init__(self):
        self.inbox = MinStack()
        self.outbox = MinStack()

    def push(self, x: int) -> None:
        self.inbox.push(x)

    def _shift(self):
        if not self.outbox.st:
            while self.inbox.st:
                v = self.inbox.top(); self.inbox.pop()
                self.outbox.push(v)

    def pop(self) -> int:
        self._shift()
        v = self.outbox.top(); self.outbox.pop()
        return v

    def getMin(self) -> int:
        cands = [s.getMin() for s in (self.inbox, self.outbox) if s.st]
        return min(cands)

q4 = Answer4MinQueue(); q4.push(3); q4.push(1); q4.push(2); assert q4.getMin() == 1 and q4.pop() == 3 and q4.getMin() == 1 and q4.pop() == 1 and q4.getMin() == 2
from collections import deque
for _ in range(2000):
    q, ref = Answer4MinQueue(), deque()
    for _ in range(30):
        if not ref or random.random() < 0.5:
            v = random.randint(-5, 5); q.push(v); ref.append(v)
        elif random.random() < 0.5:
            assert q.pop() == ref.popleft()
        else:
            assert q.getMin() == min(ref)
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [21]:
def answer_5_sliding_min(nums: List[int], k: int) -> List[int]:
    q, out = Answer4MinQueue(), []
    for i, x in enumerate(nums):
        q.push(x)
        if i >= k - 1:
            out.append(q.getMin())
            q.pop()
    return out

assert answer_5_sliding_min([1, 3, -1, -3, 5, 3, 6, 7], 3) == [-1, -3, -3, -3, 3, 3]
assert answer_5_sliding_min([4], 1) == [4]
for _ in range(2000):
    a = [random.randint(-9, 9) for _ in range(random.randint(1, 15))]
    k = random.randint(1, len(a))
    assert answer_5_sliding_min(a, k) == [min(a[i:i + k]) for i in range(len(a) - k + 1)]
print("Bài 5 PASS")

Bài 5 PASS


# 15. Tổng kết

> Ngăn xếp chỉ bỏ phần tử trên cùng → lưu **min tại thời điểm push** cùng mỗi phần tử: `push (val, min(val, min_đỉnh))`, `getMin = min_đỉnh`. Hoặc hai ngăn với `mins` chỉ push khi `val <= mins[-1]`. Mọi thao tác `O(1)`.

---

## Checklist trước khi nộp

- [ ] Không chỉ giữ một biến min?
- [ ] Hai ngăn: dùng `<=` khi push vào `mins`?
- [ ] Không khởi tạo min bằng hằng số "đủ lớn"?
- [ ] `top` trả về **giá trị**, không phải cặp?